In [1]:
import pandas as pd
import numpy as np

In [2]:
A_df = pd.read_csv("data/A.csv", header=None)
B_df = pd.read_csv("data/B.csv", header=None)
C_df = pd.read_csv("data/C.csv", header=None)

# Convert all string-looking numbers to floats
A = A_df.apply(pd.to_numeric, errors='coerce').values
B = B_df.apply(pd.to_numeric, errors='coerce').values
C = C_df.apply(pd.to_numeric, errors='coerce').values

In [3]:
A_index_df = pd.read_csv("data/index_A.csv")
B_index_df = pd.read_csv("data/index_B.csv")
C_index_df = pd.read_csv("data/index_C.csv")

## Remove Transporation

In [4]:
A_transport_df = pd.read_csv("data/Transportation_A.csv")

In [5]:
# create a dict mapping each provider name to all its indices in A_index_df
mapping = A_index_df.groupby('provider name')['index'].apply(list)

# build a single flat list of all matching indices for the foreground processes
matched_indices_transport = [
    idx
    for name in A_transport_df['provider name']
    if name in mapping
    for idx in mapping[name]
]

In [6]:
import numpy as np

# matched_indices_transport is the list of indices to remove
to_drop = np.array(sorted(set(matched_indices_transport), key=int))

# 1) Remove from A_index_df
mask_keep = ~A_index_df['index'].isin(to_drop)
A_index_df = A_index_df.loc[mask_keep].copy()

# 2) Remove corresponding rows and columns from A
A = np.delete(A, to_drop, axis=0)  # remove rows
A = np.delete(A, to_drop, axis=1)  # remove columns

# 3) Remove the same columns from B (keep rows)
B = np.delete(B, to_drop, axis=1)

# 4) Reset the index column in A_index_df
A_index_df['index'] = np.arange(len(A_index_df), dtype=int)

## Remove and aggregate Electricity

In [7]:
A_elec_df = pd.read_csv("data/Electricity_A.csv")

In [8]:
# Inputs assumed:
# A : numeric numpy array (rows x cols)
# A_index_df : DataFrame with columns ["index", "provider name", "flow name", ...]
# A_elec_df : DataFrame with column ["provider name"] listing all electricity providers
# The indices in A_index_df["index"] align with both row and column positions of A.

# 0) Build the set of electricity provider names
elec_names = set(A_elec_df['provider name'].dropna().astype(str).unique())

# 1) Find their indices in A_index_df
elec_idx = A_index_df.loc[A_index_df['provider name'].astype(str).isin(elec_names), 'index'].astype(int).unique()

# 2) Locate the mix row index (must exist)
mix_name = "Electricity Mix (Global)"
mix_rows = A_index_df.loc[A_index_df['provider name'] == mix_name, 'index'].astype(int).unique()
if len(mix_rows) == 0:
    raise ValueError("Electricity Mix (Global) not found in A_index_df['provider name'].")
mix_idx = int(mix_rows[0])

# Ensure the mix row is not purged
elec_idx_set = set(map(int, elec_idx))
elec_idx_wo_mix = sorted(elec_idx_set - {mix_idx})

# 3) Aggregate: add all electricity rows (except the mix row) into the mix row, column-wise
if len(elec_idx_wo_mix) > 0:
    # in case of NaNs
    add_block = np.nansum(A[elec_idx_wo_mix, :], axis=0)
    A[mix_idx, :] = np.nan_to_num(A[mix_idx, :]) + np.nan_to_num(add_block)

# 4) Decide what to drop
rows_to_drop = np.array(elec_idx_wo_mix, dtype=int)            # drop electricity rows except the mix row
cols_to_drop = np.array(elec_idx_wo_mix, dtype=int)            # drop electricity columns except the mix column

# (Optionally also drop the mix COLUMN; keep it if you want to retain that process as a column)
# To ALSO drop the mix column, uncomment the next line:
# cols_to_drop = np.array(sorted(elec_idx_set), dtype=int)

# 5) Remove rows/columns from A and columns from B
if rows_to_drop.size > 0:
    A = np.delete(A, rows_to_drop, axis=0)
if cols_to_drop.size > 0:
    A = np.delete(A, cols_to_drop, axis=1)
    B = np.delete(B, cols_to_drop, axis=1)

# 6) Remove the same rows from A_index_df (only rows; columns in A_index_df are metadata)
if len(elec_idx_wo_mix) > 0:
    keep_mask = ~A_index_df['index'].astype(int).isin(elec_idx_wo_mix)
    A_index_df = A_index_df.loc[keep_mask].copy()

# 7) Reset the "index" column in A_index_df to reflect 0..n-1 after deletions
A_index_df['index'] = np.arange(len(A_index_df), dtype=int)

In [9]:
# 8) Zero specific entries in A for the mix column
mix_name = "Electricity Mix (Global)"
mix_col_idx_s = A_index_df.loc[A_index_df['provider name'] == mix_name, 'index'].astype(int)

if mix_col_idx_s.empty:
    raise ValueError("Mix column not found after pruning. Did you drop the mix column?")
mix_col = int(mix_col_idx_s.iloc[0])

target_names = {"Fossil Electricity", "Clean Electricity"}
row_idxs = (
    A_index_df.loc[A_index_df['provider name'].isin(target_names), 'index']
    .astype(int)
    .to_numpy()
)

if row_idxs.size > 0:
    A[row_idxs, mix_col] = 0.0
else:
    print("Warning: no rows named 'Fossil Electricity' or 'Clean Electricity' found after pruning.")

## Quality Scenario (for recycled material from mechanical)

In [10]:
quality_scenarios_df = pd.read_csv("data/quality_scenarios.csv")

In [11]:
# Assumes the following are already in memory:
# - quality_scenarios_df  with columns: "Recycling Process", "Substitutable Virgin Process", "S1 - no limit"
# - A_index_df            with columns: "Provider name", "Index"
# - A                     as a NumPy array (your A-matrix)

# Normalize helper (case-insensitive, trim spaces)
_norm = lambda s: str(s).strip().casefold()

# Build name -> index map from A_index_df
# If A_index_df['Index'] is 1-based, uncomment the "- 1" line below instead.
name_to_idx = {
    _norm(p): int(i)
    for p, i in zip(A_index_df["provider name"], A_index_df["index"])
    # for p, i in zip(A_index_df["Provider name"], A_index_df["Index"] - 1)  # <- use this if indices are 1-based
}

# Map processes to A indices
col_idx = quality_scenarios_df["Recycling Process"].astype(str).map(_norm).map(name_to_idx)
row_idx = quality_scenarios_df["Substitutable Virgin Process"].astype(str).map(_norm).map(name_to_idx)

# Save the pair (row, col) to the "Index" column
quality_scenarios_df["index"] = list(zip(row_idx, col_idx))

# Pull values to write
vals = pd.to_numeric(quality_scenarios_df["S1 - no limit"], errors="coerce")

# Only update where both indices and value are valid
mask = row_idx.notna() & col_idx.notna() & vals.notna()
rows = row_idx[mask].astype(int).to_numpy()
cols = col_idx[mask].astype(int).to_numpy()
v    = vals[mask].to_numpy(dtype=float)

# Write into A at (row, col)
A[rows, cols] = v

# Optional diagnostics:
# print("Updated entries:", mask.sum())
# print("Unmatched Recycling Process:", quality_scenarios_df.loc[col_idx.isna(), "Recycling Process"].drop_duplicates().tolist()[:10])
# print("Unmatched Substitutable Virgin Process:", quality_scenarios_df.loc[row_idx.isna(), "Substitutable Virgin Process"].drop_duplicates().tolist()[:10])

## Setting up A design

In [12]:
import numpy as np
import pandas as pd

# --- VALIDATION ---
if not isinstance(A, np.ndarray) or A.ndim != 2:
    raise ValueError("A must be a 2D NumPy array.")
n, m = A.shape
if n != m:
    raise ValueError(f"A must be square; got {A.shape}.")
need = {"index", "provider name", "flow name"}
if not need.issubset(A_index_df.columns):
    raise ValueError(f"A_index_df missing columns: {need - set(A_index_df.columns)}")

# ensure indices cover 0..n-1 uniquely
idx = pd.to_numeric(A_index_df["index"], errors="coerce").astype("Int64")
if idx.isna().any():
    raise ValueError("A_index_df['index'] contains non-integer values.")
idx_vals = idx.astype(int).to_numpy()
if len(np.unique(idx_vals)) != len(idx_vals):
    dupes = A_index_df[A_index_df.duplicated("index", keep=False)].sort_values("index")
    raise ValueError(f"Duplicate indices in A_index_df['index']:\n{dupes}")
if idx_vals.min() != 0 or idx_vals.max() != n - 1:
    raise ValueError(f"A_index_df['index'] must span 0..{n-1}.")

# --- build label arrays aligned by index ---
providers = [""] * n
flows     = [""] * n
for _, r in A_index_df.iterrows():
    i = int(r["index"])
    providers[i] = str(r["provider name"])
    flows[i]     = str(r["flow name"])

# --- assemble (no titles, no extra columns) ---
rows = []
# first two header rows (blank first two cells, then provider and flow names)
rows.append(["", ""] + providers)
rows.append(["", ""] + flows)

# numeric block with matching row labels
for i in range(n):
    rows.append([providers[i], flows[i]] + A[i, :].tolist())

final_df = pd.DataFrame(rows)

# optional: export
# final_df.to_csv("A_full_labeled_aligned_clean.csv", index=False, header=False)
A_design_df = final_df

In [13]:
decision_variables_all = pd.read_csv("data/decision_variables_dynamic.csv")
new_flow_df = pd.read_csv("data/new_flow_roadmap.csv")

In [14]:
import pandas as pd
import numpy as np

def apply_levers_add_and_populate(
    A_design_df: pd.DataFrame,
    decision_variables_all: pd.DataFrame,
    new_flow_df: pd.DataFrame,
    activate: list,
    case_insensitive: bool = False,
    overwrite: bool = True,
):
    """
    Full pipeline:
      1) Remove rows from A_design_df where 'Input parameters names' has ANY activated lever == 1
         (match against the FIRST COLUMN of A_design_df; header rows 0,1 are preserved)
      2) From removed rows, capture UNIQUE PROVIDER NAMES (col 0)
      3) Map providers via new_flow_df (Process -> Flow name), dedupe flows, append blank rows:
            ["", <Flow name>, 0, 0, ...]
      4) Populate those new rows: for each matching Flow name in new_flow_df,
         write -1 (input) or +1 (output) at the column whose header (row 0) equals 'Process'.

    Expected layout for A_design_df:
      - Row 0: ['', ''] + provider headers across (cols 2..end)
      - Row 1: ['', ''] + flow headers across    (cols 2..end)
      - Row 2+: [provider_name, flow_name] + numeric (cols 2..end)

    Returns
    -------
    updated_df : pd.DataFrame
    report : dict with counts/diagnostics
    """

    # --------- validation ---------
    if A_design_df.shape[0] < 2 or A_design_df.shape[1] < 3:
        raise ValueError("A_design_df must have 2 header rows and >=3 columns.")
    if "Input parameters names" not in decision_variables_all.columns:
        raise ValueError("decision_variables_all must include 'Input parameters names'.")
    need_nf = {"Flow name", "Process", "Type of flow"}
    if not need_nf.issubset(new_flow_df.columns):
        raise ValueError(f"new_flow_df must include columns: {need_nf}")

    df  = A_design_df.copy()
    dva = decision_variables_all.copy()
    nf  = new_flow_df.copy()

    # --------- step 1: removal (by active levers) ---------
    lever_cols = [c for c in activate if c in dva.columns]
    if not lever_cols:
        raise ValueError(f"No requested levers found in decision_variables_all. Requested: {activate}")

    for c in lever_cols:
        dva[c] = pd.to_numeric(dva[c], errors="coerce").fillna(0).astype(int)

    # Key choice for matching (exact vs case-insensitive)
    if case_insensitive:
        norm = lambda s: str(s).strip().casefold()
        dva["_name"] = dva["Input parameters names"].map(norm)
        first_col    = df.iloc[:, 0].astype(str).map(norm)
    else:
        dva["_name"] = dva["Input parameters names"].astype(str)
        first_col    = df.iloc[:, 0].astype(str)

    to_remove_names = set(dva.loc[dva[lever_cols].sum(axis=1) > 0, "_name"])

    is_header = df.index.isin([0, 1])
    remove_mask = (~is_header) & (first_col.isin(to_remove_names))

    # capture removed providers (first column) & rows count
    removed_rows_count = int(remove_mask.sum())
    removed_providers = (
        df.loc[remove_mask, df.columns[0]].dropna().astype(str).unique().tolist()
    )
    removed_providers = sorted(set(removed_providers))

    # keep flow names too, if you want them for debugging
    removed_flows_dbg = (
        df.loc[remove_mask, df.columns[1]].dropna().astype(str).unique().tolist()
    )

    filtered_df = df.loc[~remove_mask].reset_index(drop=True)

    # --------- step 2: provider -> flow mapping, add blank rows ---------
    if case_insensitive:
        norm = lambda s: str(s).strip().casefold()
        provider_keys = set(map(norm, removed_providers))
        nf["_proc"] = nf["Process"].astype(str).map(norm)
        flows_to_add = nf.loc[nf["_proc"].isin(provider_keys), "Flow name"].astype(str).unique().tolist()
    else:
        provider_keys = set(map(str, removed_providers))
        flows_to_add = nf.loc[nf["Process"].astype(str).isin(provider_keys), "Flow name"].astype(str).unique().tolist()

    flows_to_add = sorted(set(flows_to_add))

    numeric_len = filtered_df.shape[1] - 2
    zero_row = [0.0] * numeric_len
    new_rows = [["", flow] + zero_row for flow in flows_to_add]

    if new_rows:
        add_df = pd.DataFrame(new_rows, columns=filtered_df.columns)
        with_added_df = pd.concat([filtered_df, add_df], ignore_index=True)
    else:
        with_added_df = filtered_df.copy()

    # --------- step 3: populate the newly added rows ---------
    # Build header map from row 0
    if case_insensitive:
        norm = lambda s: str(s).strip().casefold()
        headers_pretty = with_added_df.iloc[0, 2:].astype(str).tolist()
        header_keys    = [norm(h) for h in headers_pretty]
        header_to_col  = {k: 2 + i for i, k in enumerate(header_keys)}
        nf["_flow_key"] = nf["Flow name"].astype(str).map(norm)
        nf["_proc_key"] = nf["Process"].astype(str).map(norm)
        flow_key = lambda x: norm(str(x))
        proc_key = lambda x: norm(str(x))
        proc_display = lambda r: str(r["Process"])
    else:
        headers_pretty = with_added_df.iloc[0, 2:].astype(str).tolist()
        header_to_col  = {h: 2 + i for i, h in enumerate(headers_pretty)}
        nf["_flow_key"] = nf["Flow name"].astype(str)
        nf["_proc_key"] = nf["Process"].astype(str)
        flow_key = lambda x: str(x)
        proc_key = lambda x: str(x)
        proc_display = lambda r: str(r["Process"])

    # mark the newly added rows: provider label blank & index >= 2
    first_col_after = with_added_df.iloc[:, 0].astype(str)
    is_added_row = (with_added_df.index >= 2) & (first_col_after.str.strip() == "")
    target_idxs = with_added_df.index[is_added_row].tolist()

    rows_populated = 0
    skipped_no_matches = []  # flows with no new_flow_df rows
    skipped_no_column  = []  # (process, flow) when process header not found
    skipped_bad_type   = []  # (process, flow, type)

    for i in target_idxs:
        flow_label = str(with_added_df.iat[i, 1])
        if not flow_label:
            continue

        sub = nf.loc[nf["_flow_key"] == flow_key(flow_label)]
        if sub.empty:
            skipped_no_matches.append(flow_label)
            continue

        wrote_any = False
        for _, r in sub.iterrows():
            typ = str(r["Type of flow"]).strip().casefold()
            if typ == "input":
                val = +1.0
            elif typ == "output":
                val = -1.0
            else:
                skipped_bad_type.append((proc_display(r), flow_label, r["Type of flow"]))
                continue

            # find target column (row-0 header == Process)
            col_j = header_to_col.get(proc_key(r["Process"])) if case_insensitive else header_to_col.get(r["Process"])
            if col_j is None:
                skipped_no_column.append((proc_display(r), flow_label))
                continue

            current = with_added_df.iat[i, col_j]
            if overwrite or (pd.isna(current) or float(current) == 0.0):
                with_added_df.iat[i, col_j] = val
                wrote_any = True

        if wrote_any:
            rows_populated += 1

    updated_df = with_added_df.reset_index(drop=True)

    # --------- report ---------
    report = {
        "requested_levers": lever_cols,
        "removed_rows_count": removed_rows_count,
        "removed_providers_unique": len(removed_providers),
        "removed_providers": removed_providers,
        "removed_flows_dbg": removed_flows_dbg,
        "flows_to_add": flows_to_add,
        "rows_added": len(flows_to_add),
        "rows_populated": rows_populated,
        "skipped_no_matches": skipped_no_matches,
        "skipped_no_column": skipped_no_column,
        "skipped_bad_type": skipped_bad_type,
    }
    return updated_df, report


def apply_levers_and_get_matrix(
    A_design_df: pd.DataFrame,
    decision_variables_all: pd.DataFrame,
    new_flow_df: pd.DataFrame,
    activate: list,
    case_insensitive: bool = False,
    overwrite: bool = True,
):
    """
    Wrapper that:
      - runs apply_levers_add_and_populate
      - extracts the numeric submatrix from row 3, col 3 onward
      - returns: updated_df, report, numeric_subdf, numeric_matrix (np.ndarray)
    """
    updated_df, report = apply_levers_add_and_populate(
        A_design_df=A_design_df,
        decision_variables_all=decision_variables_all,
        new_flow_df=new_flow_df,
        activate=activate,
        case_insensitive=case_insensitive,
        overwrite=overwrite,
    )

    # Extract matrix from 3rd row and 3rd column onward, coerce to numeric
    A_raw = updated_df.iloc[2:, 2:]
    A_numeric = A_raw.apply(pd.to_numeric, errors="coerce")
    A_matrix = A_numeric.to_numpy()

    return updated_df, report, A_numeric, A_matrix

In [15]:
# ------------------ example call ------------------
# Options:
# Circularity
# Upstream Chemicals
# Electricity Source
# Carbon Capture
# Microplastic Treatment
# All

activated = ["Circularity", "Upstream Chemicals", "Carbon Capture", "Microplastic Treatment"]

A_design_final_df, rep, A_numeric_df, A_design_matrix = apply_levers_and_get_matrix(
    A_design_df,
    decision_variables_all,
    new_flow_df,
    activate=activated,
    case_insensitive=True,  # robust against case/space differences
    overwrite=True          # set to False to only fill zeros
)

In [16]:
n_rows, n_cols = B.shape

# -----------------------------
# Build empty dataframe with embedded headers
# size = (n_rows + 2) x (n_cols + 2)
# -----------------------------
B_design_df = pd.DataFrame(
    "",
    index=range(n_rows + 2),
    columns=range(n_cols + 2)
)

# -----------------------------
# Fill column headers (from index_A)
# row 0: provider name
# row 1: flow name
# -----------------------------
B_design_df.iloc[0, 2:] = A_index_df.sort_values("index")["provider name"].astype(str).values
B_design_df.iloc[1, 2:] = A_index_df.sort_values("index")["flow name"].astype(str).values

# -----------------------------
# Fill row headers (from index_B)
# col 0: flow name
# -----------------------------
B_design_df.iloc[2:, 0] = B_index_df.sort_values("index")["flow name"].astype(str).values

# -----------------------------
# Insert numeric B matrix
# -----------------------------
B_design_df.iloc[2:, 2:] = B

In [17]:
A_design_final_df.to_csv("results/A_design_final_df.csv", index=False)

In [18]:
B_design_df.to_csv(
    "results/B_design_final_df.csv",
    index=False,
    header=False
)

# A dynamic

In [19]:
import os
import pandas as pd

# -----------------------------
# Load inputs
# -----------------------------
a_design_df      = pd.read_csv("results/A_design_final_df.csv")
elec_mix_df      = pd.read_csv("data/electricity_mix_2025_2100_ssp1.csv")
indices_mapping  = pd.read_csv("data/iam-elec-indices.csv")

ratio_oil_ng_df  = pd.read_csv("data/ratio_oil_vs_natural_gas_2025_2100_ssp1.csv")
map_oil_ng_df    = pd.read_csv("data/oil-vs-NG-feedstock.csv")

# -----------------------------
# Prebuild lookup tables (A_design_final_df structure)
# - column provider names: row 0, cols 2:
# - row identifiers:       col 0, rows 2:
# - numeric block:         rows 2:, cols 2:
# -----------------------------
col_provider_series = a_design_df.iloc[0, 2:].astype(str)
row_provider_series = a_design_df.iloc[2:, 0].astype(str)

provider_to_col = {name: 2 + i for i, name in enumerate(col_provider_series.tolist())}
provider_to_row = {name: 2 + i for i, name in enumerate(row_provider_series.tolist())}

# -----------------------------
# Pre-index oil/NG ratio table by year
# -----------------------------
ratio_oil_ng_by_year = ratio_oil_ng_df.set_index("Year")

# -----------------------------
# Build feedstock targets from oil-vs-NG-feedstock map
# (dynamic csv name -> value in ratio table)
# Provider name -> column provider (row 0)
# Input parameters names -> row identifier (col 0)
# -----------------------------
feedstock_targets_oil_ng = []
for _, r in map_oil_ng_df.iterrows():
    dyn_name   = str(r["dynamic csv name"])
    prov_name  = str(r["Provider name"])
    input_name = str(r["Input parameters names"])

    row_idx = provider_to_row.get(input_name)
    col_idx = provider_to_col.get(prov_name)

    if (row_idx is not None) and (col_idx is not None):
        feedstock_targets_oil_ng.append((dyn_name, row_idx, col_idx))

# -----------------------------
# Generate per-year dataframes
# -----------------------------
yearly_dataframes = {}

for _, year_row in elec_mix_df.iterrows():
    year = int(year_row["Year"])
    df_year = a_design_df.copy()

    # (a) Electricity mix updates
    for _, m in indices_mapping.iterrows():
        iam_source   = str(m["IAM Index"])
        col_provider = str(m["LCI Column Index"])
        row_provider = str(m["LCI Row Index"])

        if iam_source not in elec_mix_df.columns:
            continue

        r = provider_to_row.get(row_provider)
        c = provider_to_col.get(col_provider)

        if (r is None) or (c is None):
            continue

        v = year_row[iam_source]
        if pd.isna(v):
            continue

        df_year.iat[r, c] = -abs(float(v)) / 100.0  # percent -> fraction, negative input

    # (b) Oil vs Natural Gas updates (from ratio table for this year)
    if year in ratio_oil_ng_by_year.index:
        r_year = ratio_oil_ng_by_year.loc[year]
        for dyn_name, r_idx, c_idx in feedstock_targets_oil_ng:
            if dyn_name in r_year.index and pd.notna(r_year[dyn_name]):
                df_year.iat[r_idx, c_idx] = -abs(float(r_year[dyn_name]))  # negative input

    yearly_dataframes[year] = df_year

# -----------------------------
# Save outputs
# -----------------------------
outdir = "results/Technology Matrix for each year"
os.makedirs(outdir, exist_ok=True)

for year, df in yearly_dataframes.items():
    df.to_csv(f"{outdir}/A_design_{year}.csv", index=False, header=False)

# B dynamic

In [20]:
import os
import pandas as pd

# -----------------------------
# Load inputs
# -----------------------------
B_design_df = pd.read_csv("results/B_design_final_df.csv", header=None, low_memory=False)
A_design_df = pd.read_csv("results/A_design_final_df.csv", low_memory=False)

bio_map_df  = pd.read_csv("data/biogenic-landuse-carbon.csv")
land_df     = pd.read_csv("data/land-use-emissions_annual_2025_2100.csv")

# -----------------------------
# Lookups for B_design_final_df (handle duplicates)
# - providers for columns: row 0, cols 2:
# - flows for rows:        col 0, rows 2:
# -----------------------------
B_col_providers = B_design_df.iloc[0, 2:].astype(str)
B_row_flows     = B_design_df.iloc[2:, 0].astype(str)

# name -> list of column indices
provider_to_cols_B = {}
for i, name in enumerate(B_col_providers.tolist()):
    provider_to_cols_B.setdefault(name, []).append(2 + i)

# flows are usually unique; but we can make it robust too:
flow_to_rows_B = {}
for i, name in enumerate(B_row_flows.tolist()):
    flow_to_rows_B.setdefault(name, []).append(2 + i)

# -----------------------------
# Lookups for A_design_final_df (both axes are providers; handle duplicates)
# - providers for columns: row 0, cols 2:
# - providers for rows:    col 0, rows 2:
# -----------------------------
A_col_providers = A_design_df.iloc[0, 2:].astype(str)
A_row_providers = A_design_df.iloc[2:, 0].astype(str)

provider_to_cols_A = {}
for i, name in enumerate(A_col_providers.tolist()):
    provider_to_cols_A.setdefault(name, []).append(2 + i)

provider_to_rows_A = {}
for i, name in enumerate(A_row_providers.tolist()):
    provider_to_rows_A.setdefault(name, []).append(2 + i)

def A_diag_abs(provider_name: str) -> float:
    """
    If duplicates exist, return abs(A[r,c]) for the first matching diagonal pair.
    (You can change this to mean/max if you prefer.)
    """
    rows = provider_to_rows_A.get(provider_name, [])
    cols = provider_to_cols_A.get(provider_name, [])
    if not rows or not cols:
        return 1.0

    # pick first pair (or you can loop all pairs)
    r = rows[0]
    c = cols[0]
    v = A_design_df.iat[r, c]
    return abs(float(v)) if pd.notna(v) else 1.0

# -----------------------------
# SSP1 land-use emissions by year
# -----------------------------
ssp1_series = (
    land_df.loc[land_df["Scenario"].astype(str).str.upper() == "SSP1", ["Year", "Value"]]
    .assign(Year=lambda d: d["Year"].astype(int))
    .set_index("Year")["Value"]
)

# -----------------------------
# Build B dataframe for each year
# Update all duplicate provider columns in B (and all duplicate flow rows if they exist)
# -----------------------------
years = list(range(2025, 2101))
B_yearly = {}

for y in years:
    df_y = B_design_df.copy()

    if y not in ssp1_series.index:
        B_yearly[y] = df_y
        continue

    land_val = float(ssp1_series.loc[y])

    for _, r in bio_map_df.iterrows():
        flow_name = str(r["Flow"])
        prov_name = str(r["Provider"])

        row_list = flow_to_rows_B.get(flow_name, [])
        col_list = provider_to_cols_B.get(prov_name, [])

        if not row_list or not col_list:
            continue

        scale = A_diag_abs(prov_name)
        new_val = land_val * scale

        # update all duplicates
        for br in row_list:
            for bc in col_list:
                df_y.iat[br, bc] = new_val

    B_yearly[y] = df_y

# -----------------------------
# Save outputs
# -----------------------------
outdir = "results/B_matrix_for_each_year"
os.makedirs(outdir, exist_ok=True)

for y, df in B_yearly.items():
    df.to_csv(f"{outdir}/B_design_{y}.csv", index=False, header=False)

# A meta

In [21]:
import numpy as np

# numeric A_meta: dict[year] -> np.ndarray (numeric block only)
A_meta = {year: df.iloc[2:, 2:].to_numpy(dtype=float) for year, df in yearly_dataframes.items()}

# (optional) enforce full year range with base matrix if missing
years = list(range(2025, 2101))
A_base = a_design_df.iloc[2:, 2:].to_numpy(dtype=float)
for y in years:
    if y not in A_meta:
        A_meta[y] = A_base.copy()

# B meta

In [22]:
import numpy as np

# numeric B_meta: dict[year] -> np.ndarray (numeric block only)
B_meta = {year: df.iloc[2:, 2:].to_numpy(dtype=float) for year, df in B_yearly.items()}

# (optional) enforce full year range with base matrix if missing
years = list(range(2025, 2101))
B_base = B_design_df.iloc[2:, 2:].to_numpy(dtype=float)

for y in years:
    if y not in B_meta:
        B_meta[y] = B_base.copy()

# f meta

In [23]:
#f meta data
demand_df = pd.read_csv("data/packaging_production_manual_2025_2100_kg.csv")
demand = dict(zip(demand_df["Year"].astype(int), demand_df["Packaging_Plastic_kg"].astype(float)))
f_meta = {}
n_A = A.shape[0]
years = range(2025, 2101)
for y in years:
    f = np.zeros(n_A)
    f[0] = demand[y]
    f_meta[y] = f

# dynamic costs

In [24]:
financial_df = pd.read_csv("data/Financial.csv")
foreground_s_df = pd.read_csv("data/Foreground Processes Design.csv")
search_elements_foreground = foreground_s_df['provider name'].dropna().astype(str).tolist()
search_elements_finanical = financial_df['LCI Column Index'].dropna().astype(str).tolist()
positive_s_df = pd.read_csv("data/positive_s_roadmap.csv")
search_elements_positive = positive_s_df['provider name'].dropna().astype(str).tolist()
positive_s_indices = [A_design_final_df.iloc[0,:].tolist().index(elem) for elem in search_elements_positive]
positive_s_indices = [i-1 for i in positive_s_indices]
foreground_s_indices_columns = [A_design_final_df.iloc[0,:].tolist().index(elem) for elem in search_elements_foreground]
foreground_s_indices_columns = [i-1 for i in foreground_s_indices_columns]

# dynamic prices

In [25]:
financial_df = pd.read_csv("data/Financial.csv")
financial_df['LCI Column Index'] = financial_df['LCI Column Index'].astype(str)
values_dict = {}
values_dict = (
    financial_df
    .loc[financial_df['LCI Column Index'].isin(search_elements_finanical), ['LCI Column Index', 'Value']]
    .set_index('LCI Column Index')['Value']
    .to_dict()
)
result_dict = {}

# ensure consistent string comparison
A_design_final_df.iloc[:, 0] = A_design_final_df.iloc[:, 0].astype(str)

for i, name in enumerate(A_design_final_df.iloc[:, 0]):
    if name in values_dict:
        result_dict[i - 1] = values_dict[name]

inflation_rate = 0.025
years = range(2025, 2101)

cost_2025 = result_dict  # your existing dictionary

cost_by_year = {
    (i, y): v * (1 + inflation_rate) ** (y - 2025)
    for i, v in cost_2025.items()
    for y in years
}

# variable cost ratios

In [26]:
f_voc_df = pd.read_csv("data/f_voc_every_line_use_collection.csv")

# make sure keys match
f_voc_df["Process"] = f_voc_df["Process"].astype(str)
A_design_final_df.iloc[0, :] = A_design_final_df.iloc[0, :].astype(str)

# keep only what you need, indexed by Process (so lookup is easy)
lookup = (
    f_voc_df.loc[
        f_voc_df["Process"].isin(search_elements_foreground),
        ["Process", "f_VOC_of_total_cost", "TRL", "k values", "lambda values"]
    ]
    .drop_duplicates(subset=["Process"])
    .set_index("Process")
)

f_voc_dict = {}
init_trl_dict = {}
k_values_dict = {}
lambda_values_dict = {}

for i, name in enumerate(A_design_final_df.iloc[0, :].astype(str)):
    if name in lookup.index:
        idx = i - 1  # your existing indexing convention
        f_voc_dict[idx]         = lookup.at[name, "f_VOC_of_total_cost"]
        init_trl_dict[idx]       = lookup.at[name, "TRL"]
        k_values_dict[idx]       = lookup.at[name, "k values"]
        lambda_values_dict[idx] = lookup.at[name, "lambda values"]

In [27]:
foreground_set = set(foreground_s_indices_columns)

new_dict_f_voc = {
    i: f_voc_dict.get(i, 1)
    for i in foreground_set
    if i in foreground_set
    if i in f_voc_dict or i not in f_voc_dict
}

new_dict_init_trl = {
    i: init_trl_dict.get(i, 10)
    for i in foreground_set
    if i in foreground_set
    if i in init_trl_dict or i not in init_trl_dict
}

new_dict_k_values = {
    i: k_values_dict.get(i, 0)
    for i in foreground_set
    if i in foreground_set
    if i in k_values_dict or i not in k_values_dict
}

new_dict_lambda_values = {
    i: lambda_values_dict.get(i, 0)
    for i in foreground_set
    if i in foreground_set
    if i in lambda_values_dict or i not in lambda_values_dict
}

In [28]:
trl_min = 1
trl_max = 10

new_dict_init_trl_norm = {
    k: (v - trl_min) / (trl_max - trl_min)
    for k, v in new_dict_init_trl.items()
}

# Optimization

In [41]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy
from pyomo.environ import *
from pyomo.environ import RangeSet
from pyomo.environ import value
import plotly.graph_objects as go

In [ ]:
foreground_indices

In [47]:
import math
from pyomo.environ import (
    ConcreteModel, RangeSet, Set, Param, Var, Constraint, Objective,
    Reals, NonNegativeReals, Binary, minimize, value
)

# -----------------------
# SETTINGS
# -----------------------
t0 = 2025
t_start = 2025
t_end = 2100

TRL_CUTOFF = 8
trl_norm_cutoff = (TRL_CUTOFF - 1) / 9.0  # TRL8 => 7/9

TRL_CAP = 1.0
RATIO_CAP = 0.8
RATIO_EPS = 1e-12

# Big-M must be a safe upper bound on s (for years where tech is allowed)
M_S = 1e20

# Optional: force "if selected => used at least once"
EPS_USE = 0.0

# Optional: tiny penalty on selecting technologies
SELECT_PENALTY = 0.0


# -----------------------
# ONE-TIME R&D COST (based on initial TRL integer at 2025)
# -----------------------
def rd_cost_from_trl(trl_int):
    trl = int(trl_int)
    if trl < 3:
        return 111_100_000.0
    elif trl < 5:
        return 111_000_000.0
    elif trl < 6:
        return 110_000_000.0
    elif trl < 8:
        return 100_000_000.0
    else:
        return 0.0

# ============================================================
# PRECOMPUTE TRL*(t) AND r*(t)
# ============================================================
C_max = A_design_matrix.shape[1]

# You said set_L is your foreground process indices; we will use ONLY set_L for:
# - binaries y[j]
# - TRL gating
# - ratio evolution
# So L must be consistent with A columns.
foreground_indices = sorted([j for j in new_dict_init_trl.keys() if 1 <= j <= C_max])

# Precompute TRL allow (0/1) and r*(t)
allow_data = {}   # (j,t) -> 0/1 (TRL threshold)
ratio_data = {}   # (j,t) -> r*(t)
rd_cost_data = {} # j -> RD cost

for j in foreground_indices:
    # ---- TRL* inputs
    y0_trl = float(new_dict_init_trl_norm.get(j, 1.0))   # TRL*(2025)
    k_j    = float(new_dict_k_values.get(j, 0.0))        # k parameter

    # ---- ratio inputs
    y0_r   = float(new_dict_f_voc.get(j, 1.0))           # r*(2025)
    lam_j  = float(new_dict_lambda_values.get(j, 0.0))   # lambda parameter

    # ---- RD cost based on initial TRL integer at 2025
    rd_cost_data[j] = rd_cost_from_trl(new_dict_init_trl.get(j, 10))

    for t in range(t_start, t_end + 1):
        dt = float(t - t0)

        # ---------- TRL*(t): your exact formula + cap at 1.0 ----------
        trl_t = y0_trl + 1.0 / (1.0 + math.exp(-k_j * dt))
        if trl_t > TRL_CAP:
            trl_t = TRL_CAP
        allow_data[(j, t)] = 1.0 if trl_t >= trl_norm_cutoff else 0.0

        # ---------- r*(t): your exact formula + cap at 0.8 ----------
        # If r*(2025) already >= 0.8, it never changes.
        if y0_r >= RATIO_CAP:
            r_t = y0_r
        else:
            r_t = y0_r + 1.0 / (1.0 + math.exp(-lam_j * dt))
            if r_t > RATIO_CAP:
                r_t = RATIO_CAP

        ratio_data[(j, t)] = max(RATIO_EPS, float(r_t))


# ============================================================
# BUILD PYOMO MODEL
# ============================================================
model = ConcreteModel()

# -----------------------
# SETS
# -----------------------
model.set_C = RangeSet(1, A_design_matrix.shape[1])
model.set_R = RangeSet(1, A_design_matrix.shape[0])
model.set_T = Set(initialize=range(t_start, t_end + 1), ordered=True)

model.set_Q = Set(initialize=positive_s_indices)

# k indices used in unit_cost
model.set_K = Set(initialize=sorted({k for (k, t) in cost_by_year.keys()}), ordered=True)

# Foreground indices (your main tech set)
model.set_L = Set(initialize=foreground_indices, ordered=True)

# -----------------------
# PARAMETERS
# -----------------------
model.unit_cost = Param(model.set_K, model.set_T, initialize=cost_by_year, within=Reals)

def A_init_rule(m, r, c, t):
    return float(A_meta[t][r - 1, c - 1])

def f_init_rule(m, r, t):
    return float(f_meta[t][r - 1])

model.A = Param(model.set_R, model.set_C, model.set_T, initialize=A_init_rule, within=Reals)
model.f = Param(model.set_R, model.set_T, initialize=f_init_rule, within=Reals)

def allow_init_rule(m, j, t):
    return float(allow_data[(j, t)])

model.allow = Param(model.set_L, model.set_T, initialize=allow_init_rule, within=Reals, mutable=False)

def ratio_init_rule(m, j, t):
    return float(ratio_data[(j, t)])

model.voc_ratio = Param(model.set_L, model.set_T, initialize=ratio_init_rule, within=Reals, mutable=False)

def rd_cost_init_rule(m, j):
    return float(rd_cost_data[j])

model.rd_cost = Param(model.set_L, initialize=rd_cost_init_rule, within=Reals, mutable=False)

# -----------------------
# VARIABLES
# -----------------------
# scaling factors (can be negative in general, but we will force s>=0 for j in set_L)
model.s = Var(model.set_C, model.set_T, within=Reals)

# binary: whether tech is invested/allowed at all
model.y = Var(model.set_L, within=Binary)

# -----------------------
# CONSTRAINTS
# -----------------------
# Balance: A*s = f (year-by-year)
def balance_rule(m, r, t):
    return sum(m.A[r, c, t] * m.s[c, t] for c in m.set_C) == m.f[r, t]
model.balance = Constraint(model.set_R, model.set_T, rule=balance_rule)

# Positive scaling for subset Q (your original constraint)
def positive_scale_rule(m, i, t):
    return m.s[i, t] >= 0
model.positive_scale = Constraint(model.set_Q, model.set_T, rule=positive_scale_rule)

# NEW: enforce s>=0 for all foreground indices in cost (prevents negative-cost gaming)
def nonneg_foreground_rule(m, j, t):
    return m.s[j, t] >= 0
model.nonneg_foreground = Constraint(model.set_L, model.set_T, rule=nonneg_foreground_rule)

# Selection gate (if y[j]=0 then s[j,t]=0)
def select_gate_rule(m, j, t):
    return m.s[j, t] <= M_S * m.y[j]
model.select_gate = Constraint(model.set_L, model.set_T, rule=select_gate_rule)

# TRL gate (if allow[j,t]=0 then s[j,t]=0 even if selected)
def trl_gate_rule(m, j, t):
    return m.s[j, t] <= M_S * m.y[j] * m.allow[j, t]
model.trl_gate = Constraint(model.set_L, model.set_T, rule=trl_gate_rule)

# Optional: if selected, must be used at least once (summing s since s>=0 here)
if EPS_USE and EPS_USE > 0.0:
    def must_use_rule(m, j):
        return sum(m.s[j, t] for t in m.set_T) >= EPS_USE * m.y[j]
    model.must_use = Constraint(model.set_L, rule=must_use_rule)

# -----------------------
# OBJECTIVE COEFFICIENTS
# -----------------------
# coef[j,t] = (1/r*(j,t)) * sum_k |A_meta[t][k-1, j-1]| * unit_cost[k,t]
def coef_init_rule(m, j, t):
    r_t = float(value(m.voc_ratio[j, t]))
    if abs(r_t) < RATIO_EPS:
        r_t = RATIO_EPS

    return (1.0 / r_t) * sum(
        abs(A_meta[t][k - 1, j - 1]) * float(value(m.unit_cost[k, t]))
        for k in m.set_K
    )

model.coef = Param(model.set_L, model.set_T, initialize=coef_init_rule, within=Reals, mutable=False)

# -----------------------
# OBJECTIVE (NO ABS)
# -----------------------
# Operating cost + one-time RD cost (if selected)
model.obj = Objective(
    expr=
        sum(model.coef[j, t] * model.s[j, t] for t in model.set_T for j in model.set_L)
        + sum(model.rd_cost[j] * model.y[j] for j in model.set_L)
        + (SELECT_PENALTY * sum(model.y[j] for j in model.set_L) if SELECT_PENALTY else 0.0),
    sense=minimize
)

# -----------------------
# SOLVE (example)
# -----------------------
# from pyomo.opt import SolverFactory
# solver = SolverFactory("gurobi")
# results = solver.solve(model, tee=True)
# print(results.solver.termination_condition)


In [48]:

# --- SOLVE ---
solver = SolverFactory("gurobi")   # or "cbc" if you have it installed
results = solver.solve(model, tee=True)

Set parameter Username
Academic license - for non-commercial use only - expires 2026-12-24
Read LP format model from file /var/folders/q1/dbyqy0pj3lv2d5259l03mn_00000gn/T/tmpnxhl0y_3.pyomo.lp
Reading time = 0.31 seconds
x54756: 155193 rows, 54756 columns, 423957 nonzeros
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[rosetta2])

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 155193 rows, 54756 columns and 423957 nonzeros
Model fingerprint: 0xf34db75f
Variable types: 54417 continuous, 339 integer (339 binary)
Coefficient statistics:
  Matrix range     [3e-11, 1e+20]
  Objective range  [9e-03, 1e+08]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+12]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerical issues.
Presolve removed 135356 rows and 34002 columns
Presolve time: 0.27s
Presolved: 19837 rows, 20754 columns, 127226 nonzeros
Variable types:

In [49]:
import pandas as pd
from pyomo.environ import value

# 1. Load the CSV metadata (without assuming headers)
df_meta = pd.read_csv('results/A_design_final_df.csv', header=None, low_memory=False)

# 2. Extract Process Names from the first descriptive row
# We found "Use & Collection" is in Row 1, Column 2.
# This slice (from index 2 to the end) provides exactly 689 labels.
process_names = df_meta.iloc[1, 2:].tolist()

# 3. Extract solution values from the Pyomo model
# Dictionary keys: (column_index, year), values: optimized s
s_results = {
    (c, t): value(model.s[c, t]) 
    for c in model.set_C 
    for t in model.set_T
}

# 4. Create the DataFrame (Index: Years, Columns: Process Indices)
df_s = pd.Series(s_results).unstack(level=0)

# 5. Apply the labels
# df_s.columns initially contains 1, 2, ..., 689
# process_names contains "Use & Collection", "Pyrolysis...", etc.

if len(df_s.columns) == len(process_names):
    df_s.columns = process_names
    print("Success: Columns labeled correctly.")
else:
    print(f"Size Mismatch: Model has {len(df_s.columns)} vars, but found {len(process_names)} labels.")

# 6. Display the matrix
print("\nFinal Solution Matrix (Years x Processes):")
print(df_s.head())

# Optional: Export to CSV
df_s.to_csv("results/model_solutions_labeled.csv", index=False)

Success: Columns labeled correctly.

Final Solution Matrix (Years x Processes):
      Use & Collection  \
2025      2.200000e+11   
2026      2.280000e+11   
2027      2.360000e+11   
2028      2.440000e+11   
2029      2.520000e+11   

      Pyrolysis, LDPE Food Packaging Film (med-large format)  \
2025                                                0.0        
2026                                                0.0        
2027                                                0.0        
2028                                                0.0        
2029                                                0.0        

      Mechanical Recycling, LDPE Food Packaging Film (med-large format) (Flakes)  \
2025                                       1.466206e+10                            
2026                                       1.519523e+10                            
2027                                       1.572839e+10                            
2028                                      